In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score,
    recall_score, make_scorer
)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from pathlib import Path
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate


In [2]:
# === CONFIGURACION ===
REPO_ROOT = Path(".").resolve()
RUN_ID = datetime.today().strftime("%Y%m%d")

INPUT_BASE = REPO_ROOT / "data" / "intermediate" / "05_seleccion_v2"
OUTPUT_BASE = REPO_ROOT / "reports" / "10_modelo_redmlp_v2" / RUN_ID
RESUME_OUTPUT_PATH = REPO_ROOT / "reports" / "10_modelo_redmlp_v2" / "20260505" / "RedMLP_v2_01_2026-05-05"
ONLY_TARGETS = ["5_vs_resto"]


MODEL_NAME = "RedMLP_v2"
INTENTO = 1
N_SPLITS = 3
TARGET_CLASS = 1  # clase positiva binaria; la clase nula es 0
MAX_ITER = 60
LEARNING_RATE_INIT = 0.002
fecha_actual = datetime.today().strftime("%Y-%m-%d")

if not INPUT_BASE.exists():
    raise FileNotFoundError("No se encontro data/intermediate/05_seleccion_v2")

input_candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
if not input_candidates:
    raise FileNotFoundError("No se encontraron subcarpetas en data/intermediate/05_seleccion_v2")
INPUT_PATH = input_candidates[-1]
print(f"Usando input: {INPUT_PATH}")

TARGET_DIRS = sorted([p.name for p in INPUT_PATH.iterdir() if p.is_dir()])
if ONLY_TARGETS:
    missing_targets = sorted(set(ONLY_TARGETS) - set(TARGET_DIRS))
    if missing_targets:
        raise FileNotFoundError(f"Targets pedidos no existen en {INPUT_PATH}: {missing_targets}")
    TARGET_DIRS = ONLY_TARGETS
if not TARGET_DIRS:
    raise FileNotFoundError(f"No se encontraron targets dentro de {INPUT_PATH}")
print(f"Targets detectados: {TARGET_DIRS}")

if RESUME_OUTPUT_PATH is not None and Path(RESUME_OUTPUT_PATH).exists():
    OUTPUT_PATH = Path(RESUME_OUTPUT_PATH)
    print(f"Modo resume: usando output existente {OUTPUT_PATH}")
else:
    OUTPUT_PATH = OUTPUT_BASE / f"{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

BALANCE_METHODS = {
    "NONE": None,
    "SMOTE": SMOTE(random_state=42),
    "UNDER": RandomUnderSampler(random_state=42),
    "SMOTEENN": SMOTEENN(random_state=42)
}

metricas_totales = []


Usando input: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\05_seleccion_v2\05_2026_03_31
Targets detectados: ['5_vs_resto']
Modo resume: usando output existente C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\10_modelo_redmlp_v2\20260505\RedMLP_v2_01_2026-05-05


In [3]:
def build_mlp(num_classes: int):
    "MLP sencillo con escalado previo, pensado para muchas variantes."
    hidden = (96, 48) if num_classes > 2 else (64, 32)
    return MLPClassifier(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        learning_rate_init=LEARNING_RATE_INIT,
        max_iter=MAX_ITER,
        batch_size=128,
        alpha=1e-4,
        early_stopping=True,
        n_iter_no_change=6,
        validation_fraction=0.12,
        shuffle=True,
        random_state=42,
        verbose=False,
    )


def build_pipeline(balancer, num_classes: int):
    steps = []
    if balancer is not None:
        steps.append(("balance", balancer))
    steps.append(("scale", StandardScaler(with_mean=False)))
    steps.append(("model", build_mlp(num_classes)))
    return Pipeline(steps)


def resolve_target_class(y, target):
    classes = list(pd.Series(y).unique())
    if target in classes:
        return target
    if str(target) in [str(c) for c in classes]:
        for c in classes:
            if str(c) == str(target):
                return c
    raise ValueError(f"TARGET_CLASS {target} no esta en clases disponibles: {classes}")


def update_best(best_score, best_info, resumen_cv, base_name):
    cv_recall_target = resumen_cv["cv_recall_target"]
    cv_macro_f1 = resumen_cv["cv_macro_f1"]
    if (cv_recall_target > best_score) or (
        cv_recall_target == best_score
        and cv_macro_f1 > (best_info.get("cv_macro_f1", -1.0) if best_info else -1.0)
    ):
        best_score = cv_recall_target
        best_info = {
            "target": resumen_cv["target"],
            "balanceo": resumen_cv["balanceo"],
            "base_name": base_name,
            "target_class": resumen_cv["target_class"],
            "target_class_adj": resumen_cv["target_class_adj"],
            "cv_recall_target": cv_recall_target,
            "cv_macro_f1": cv_macro_f1,
        }
    return best_score, best_info


for target_name in TARGET_DIRS:
    print(f"\n===== Target: {target_name} =====")
    target_input_path = INPUT_PATH / target_name
    target_output_path = OUTPUT_PATH / target_name
    target_output_path.mkdir(parents=True, exist_ok=True)

    variantes_X = sorted([f.name for f in target_input_path.iterdir() if f.is_file() and f.name.startswith("X_train_")])
    print(f"Se detectaron {len(variantes_X)} variantes de X_train_* en {target_input_path}")
    if not variantes_X:
        print("  No hay variantes para procesar; se omite target.")
        continue

    best_score = -1.0
    best_info = None
    metricas_target = []

    for balance_name, balancer in BALANCE_METHODS.items():
        print(f"=== Balanceo: {balance_name} ===")
        output_subdir = target_output_path / balance_name
        output_subdir.mkdir(parents=True, exist_ok=True)

        for x_file in variantes_X:
            base_name = x_file.replace("X_train_", "").replace(".parquet", "")
            try:
                print(f"Procesando: {base_name}")
                cv_path = output_subdir / f"cv_metricas_{base_name}.csv"
                if cv_path.exists():
                    df_cv = pd.read_csv(cv_path)
                    resumen_cv = {
                        "target": target_name,
                        "modelo": base_name,
                        "balanceo": balance_name,
                        "target_class": df_cv["target_class"].iloc[0] if "target_class" in df_cv.columns else TARGET_CLASS,
                        "target_class_adj": df_cv["target_class_adj"].iloc[0] if "target_class_adj" in df_cv.columns else TARGET_CLASS,
                        "cv_recall_target": float(df_cv["cv_recall_target"].mean()),
                        "cv_macro_f1": float(df_cv["cv_macro_f1"].mean()),
                        "nan_total_train": int(df_cv["nan_total_train"].iloc[0]) if "nan_total_train" in df_cv.columns else 0,
                        "nan_total_test": int(df_cv["nan_total_test"].iloc[0]) if "nan_total_test" in df_cv.columns else 0,
                    }
                    metricas_target.append(resumen_cv)
                    metricas_totales.append(resumen_cv)
                    best_score, best_info = update_best(best_score, best_info, resumen_cv, base_name)
                    print(f"  Reusado CV existente: recall clase {resumen_cv['target_class']}: {resumen_cv['cv_recall_target']:.3f} | CV macro F1: {resumen_cv['cv_macro_f1']:.3f}")
                    continue

                X_train = pd.read_parquet(target_input_path / f"X_train_{base_name}.parquet").astype("float32")
                X_test = pd.read_parquet(target_input_path / f"X_test_{base_name}.parquet").astype("float32")
                y_train = pd.read_parquet(target_input_path / f"y_train_{base_name}.parquet").squeeze()
                y_test = pd.read_parquet(target_input_path / f"y_test_{base_name}.parquet").squeeze()

                nan_total_train = int(X_train.isna().sum().sum())
                nan_total_test = int(X_test.isna().sum().sum())
                if nan_total_train > 0 or nan_total_test > 0:
                    print(f"  NaN - train: {nan_total_train}, test: {nan_total_test}")

                if pd.Series(y_train).nunique() < 2:
                    print("  Dataset con una sola clase en train; se omite.")
                    continue

                target_class = resolve_target_class(y_train, TARGET_CLASS)
                y_min = y_train.min()
                y_train_adj = y_train - y_min
                y_test_adj = y_test - y_min
                target_class_adj = target_class - y_min
                num_classes = int(pd.Series(y_train_adj).nunique())

                cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
                model_cv = build_pipeline(balancer, num_classes)

                target_scorer = make_scorer(
                    recall_score,
                    labels=[target_class_adj],
                    average="macro",
                    zero_division=0
                )
                scorers = {
                    "recall_target": target_scorer,
                    "f1_macro": "f1_macro"
                }
                cv_results = cross_validate(
                    model_cv,
                    X_train,
                    y_train_adj,
                    cv=cv,
                    scoring=scorers,
                    n_jobs=1
                )

                cv_recall_scores = cv_results["test_recall_target"]
                cv_macro_f1_scores = cv_results["test_f1_macro"]
                cv_recall_target = float(np.mean(cv_recall_scores))
                cv_macro_f1 = float(np.mean(cv_macro_f1_scores))

                df_cv = pd.DataFrame({
                    "target": target_name,
                    "fold": range(1, len(cv_recall_scores) + 1),
                    "cv_recall_target": cv_recall_scores,
                    "cv_macro_f1": cv_macro_f1_scores,
                    "nan_total_train": [nan_total_train] * len(cv_recall_scores),
                    "nan_total_test": [nan_total_test] * len(cv_recall_scores),
                    "target_class": [target_class] * len(cv_recall_scores),
                    "target_class_adj": [target_class_adj] * len(cv_recall_scores)
                })
                df_cv.to_csv(cv_path, index=False)

                resumen_cv = {
                    "target": target_name,
                    "modelo": base_name,
                    "balanceo": balance_name,
                    "target_class": target_class,
                    "target_class_adj": target_class_adj,
                    "cv_recall_target": cv_recall_target,
                    "cv_macro_f1": cv_macro_f1,
                    "nan_total_train": nan_total_train,
                    "nan_total_test": nan_total_test
                }
                metricas_target.append(resumen_cv)
                metricas_totales.append(resumen_cv)
                best_score, best_info = update_best(best_score, best_info, resumen_cv, base_name)

                print(f"  CV recall clase {target_class}: {cv_recall_target:.3f} | CV macro F1: {cv_macro_f1:.3f}")

            except Exception as e:
                print(f" Error en {base_name} con balanceo {balance_name}: {e}")

    if best_info is None:
        print(f"No se pudo seleccionar un modelo ganador para {target_name}; revisa los logs.")
        continue

    print(f"\nMejor modelo por CV para {target_name} (recall clase {best_info['target_class']}): {best_info}")

    balance_name = best_info["balanceo"]
    base_name = best_info["base_name"]
    balancer = BALANCE_METHODS[balance_name]

    output_subdir = target_output_path / balance_name
    output_subdir.mkdir(parents=True, exist_ok=True)

    X_train = pd.read_parquet(target_input_path / f"X_train_{base_name}.parquet").astype("float32")
    X_test = pd.read_parquet(target_input_path / f"X_test_{base_name}.parquet").astype("float32")
    y_train = pd.read_parquet(target_input_path / f"y_train_{base_name}.parquet").squeeze()
    y_test = pd.read_parquet(target_input_path / f"y_test_{base_name}.parquet").squeeze()

    y_min = y_train.min()
    y_train_adj = y_train - y_min
    y_test_adj = y_test - y_min
    num_classes = int(pd.Series(y_train_adj).nunique())

    start = time.time()
    model = build_pipeline(balancer, num_classes)
    model.fit(X_train, y_train_adj)
    tiempo = (time.time() - start) / 60

    y_pred_train_adj = model.predict(X_train)
    y_pred_test_adj = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    y_pred_train = y_pred_train_adj + y_min
    y_pred_test = y_pred_test_adj + y_min

    class_order_adj = model.named_steps["model"].classes_
    class_order = class_order_adj + y_min

    report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
    report_train = pd.DataFrame(classification_report(y_train, y_pred_train, output_dict=True, zero_division=0)).T
    report_test["set"] = "test"
    report_train["set"] = "train"
    full_report = pd.concat([report_train, report_test])
    full_report["tiempo_min"] = tiempo
    full_report["target"] = target_name
    full_report.to_csv(output_subdir / f"metricas_{base_name}_BEST.csv")

    cm = confusion_matrix(y_test, y_pred_test, labels=class_order)
    with PdfPages(output_subdir / f"reporte_{base_name}_BEST.pdf") as pdf:
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap="Blues")
        plt.title(f"Matriz de Confusion - {target_name}")
        pdf.savefig(); plt.close()

        if np.unique(y_test).size >= 2:
            y_bin = label_binarize(y_test_adj, classes=class_order_adj)
            if y_bin.shape[1] == 1:
                y_bin = np.hstack([1 - y_bin, y_bin])
            proba_for_curves = y_proba
            if proba_for_curves.shape[1] == 1:
                proba_for_curves = np.hstack([1 - y_proba, y_proba])

            plt.figure()
            for i, clase in enumerate(class_order):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={roc_auc:.2f})")
            plt.plot([0, 1], [0, 1], "k--")
            plt.title(f"Curvas ROC por clase - {target_name}")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            pdf.savefig(); plt.close()

            plt.figure()
            for i, clase in enumerate(class_order):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
                pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
                plt.plot(recall, precision, label=f"Clase {clase} (PR AUC={pr_auc:.2f})")
            plt.title(f"Curvas Precision-Recall por clase - {target_name}")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.legend()
            pdf.savefig(); plt.close()
        else:
            print("  Test con 1 clase; se omiten curvas ROC/PR.")

    resumen = {
        "target": target_name,
        "modelo": base_name,
        "balanceo": balance_name,
        "target_class": best_info["target_class"],
        "target_class_adj": best_info["target_class_adj"],
        "accuracy_test": accuracy_score(y_test, y_pred_test),
        "macro_f1_test": report_test.loc["macro avg", "f1-score"],
        "weighted_f1_test": report_test.loc["weighted avg", "f1-score"],
        "accuracy_train": accuracy_score(y_train, y_pred_train),
        "macro_f1_train": report_train.loc["macro avg", "f1-score"],
        "weighted_f1_train": report_train.loc["weighted avg", "f1-score"],
        "tiempo_min": tiempo,
        "sobreajuste_macro_f1": report_train.loc["macro avg", "f1-score"] - report_test.loc["macro avg", "f1-score"],
        "cv_recall_target": best_info["cv_recall_target"],
        "cv_macro_f1": best_info["cv_macro_f1"]
    }
    for clase in report_test.index:
        if clase not in ["accuracy", "macro avg", "weighted avg"]:
            resumen[f"f1_{clase}_test"] = report_test.loc[clase, "f1-score"]
            resumen[f"recall_{clase}_test"] = report_test.loc[clase, "recall"]
            resumen[f"precision_{clase}_test"] = report_test.loc[clase, "precision"]
            resumen[f"f1_{clase}_train"] = report_train.loc[clase, "f1-score"]
            resumen[f"recall_{clase}_train"] = report_train.loc[clase, "recall"]
            resumen[f"precision_{clase}_train"] = report_train.loc[clase, "precision"]

    metricas_target.append(resumen)
    metricas_totales.append(resumen)
    print(f"\nBEST {target_name} | {base_name} [{balance_name}]: recall_target_cv={best_info['cv_recall_target']:.3f}, F1_macro_test={resumen.get('macro_f1_test', 0):.3f}")

    df_metricas_target = pd.DataFrame(metricas_target)
    df_metricas_target.to_csv(target_output_path / "resumen_modelos_redmlp.csv", index=False)
    print(f"Se guardo resumen_modelos_redmlp.csv con {len(df_metricas_target)} filas en {target_output_path}")

if metricas_totales:
    df_metricas = pd.DataFrame(metricas_totales)
    df_metricas.to_csv(OUTPUT_PATH / "resumen_modelos_redmlp_global.csv", index=False)
    print(f"Se guardo resumen_modelos_redmlp_global.csv con {len(df_metricas)} filas en {OUTPUT_PATH}")
    display(df_metricas.head())
else:
    print("No se genero ninguna metrica; revisa los logs de arriba.")



===== Target: 5_vs_resto =====
Se detectaron 59 variantes de X_train_* en C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\05_seleccion_v2\05_2026_03_31\5_vs_resto
=== Balanceo: NONE ===
Procesando: MaxAbs_FE_ALL
  Reusado CV existente: recall clase 1: 0.757 | CV macro F1: 0.844
Procesando: MaxAbs_FE_ANOVA
  Reusado CV existente: recall clase 1: 0.261 | CV macro F1: 0.615
Procesando: MaxAbs_FE_MI
  Reusado CV existente: recall clase 1: 0.228 | CV macro F1: 0.594
Procesando: MaxAbs_FE_PCA30
  Reusado CV existente: recall clase 1: 0.678 | CV macro F1: 0.794
Procesando: MaxAbs_FE_PCAopt
  Reusado CV existente: recall clase 1: 0.768 | CV macro F1: 0.846
Procesando: MaxAbs_FE_VAR
  Reusado CV existente: recall clase 1: 0.734 | CV macro F1: 0.816
Procesando: MaxAbs_ORIGINAL_ALL
  Reusado CV existente: recall clase 1: 0.756 | CV macro F1: 0.845
Procesando: MaxAbs_ORIGINAL_ANOVA
  Reusado CV existente: recall clase 1: 0.241 | CV macro F1: 0.596
Procesando: MaxA

c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


  CV recall clase 1: 0.314 | CV macro F1: 0.605
Procesando: Robust_FE_MI
  CV recall clase 1: 0.565 | CV macro F1: 0.681
Procesando: Robust_FE_PCA30
  CV recall clase 1: 0.902 | CV macro F1: 0.751
Procesando: Robust_FE_PCAopt
  CV recall clase 1: 0.908 | CV macro F1: 0.801
Procesando: Robust_FE_VAR
  CV recall clase 1: 0.895 | CV macro F1: 0.757
Procesando: Robust_ORIGINAL_ALL
  CV recall clase 1: 0.920 | CV macro F1: 0.753
Procesando: Robust_ORIGINAL_ANOVA
  CV recall clase 1: 0.279 | CV macro F1: 0.597
Procesando: Robust_ORIGINAL_MI
  CV recall clase 1: 0.311 | CV macro F1: 0.566
Procesando: Robust_ORIGINAL_PCA30
  CV recall clase 1: 0.871 | CV macro F1: 0.736
Procesando: Robust_ORIGINAL_PCAopt
  CV recall clase 1: 0.896 | CV macro F1: 0.804
Procesando: Robust_ORIGINAL_VAR
  CV recall clase 1: 0.912 | CV macro F1: 0.706
Procesando: Standard_FE_ALL
  CV recall clase 1: 0.914 | CV macro F1: 0.801
Procesando: Standard_FE_ANOVA
  CV recall clase 1: 0.274 | CV macro F1: 0.600
Procesando: 

,target,modelo,balanceo,target_class,target_class_adj,cv_recall_target,cv_macro_f1,nan_total_train,nan_total_test,accuracy_test,...,precision_0_test,f1_0_train,recall_0_train,precision_0_train,f1_1_test,recall_1_test,precision_1_test,f1_1_train,recall_1_train,precision_1_train
0,5_vs_resto,MaxAbs_FE_ALL,NONE,1,1,0.757191,0.844297,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5_vs_resto,MaxAbs_FE_ANOVA,NONE,1,1,0.260974,0.614539,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5_vs_resto,MaxAbs_FE_MI,NONE,1,1,0.228408,0.593873,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5_vs_resto,MaxAbs_FE_PCA30,NONE,1,1,0.678372,0.794312,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5_vs_resto,MaxAbs_FE_PCAopt,NONE,1,1,0.767652,0.845838,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
